# 13 — Past-only temporal tracking diagnostic (single-match pilot)

**Aim:** Before modelling defender reach or ball travel, verify that past-only tracking observations permit defensible velocity estimation at possession starts. Pilot on **match 1874553**. Do **not** interpret these measurements as calibrated sprint capability, interception probability, pass speed, or feasible passing opportunities.

**Source:** canonical decisions from Notebook 05 and original `tracking_extrapolated.jsonl`, nominally 10 Hz. Use **only** `t0`, `t0-5 frames`, and `t0-10 frames` (0.5 and 1.0 seconds back). Require same period and exact frame keys: never interpolate a missing frame or carry a player across periods. Distinguish *any finite extrapolated coordinates* from **all endpoints directly detected**.

**Outputs:** `outputs/P2_temporal_motion_pilot_match_1874553.jsonl` (local detail) and `outputs/P2_temporal_motion_gate.json` (upload this compact report).

The tracking data may be extrapolated; detection at two endpoints does not guarantee a continuous observed path. This diagnostic does not establish temporal continuity through unobserved intermediate frames.


In [ ]:
from pathlib import Path
from collections import Counter,defaultdict
from datetime import datetime,timezone
import json,math,statistics

ROOT=next((p for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents) if (p/'data'/'matches').is_dir()),None)
if ROOT is None:raise FileNotFoundError('Open this notebook inside your SkillCorner repo.')
OUT=ROOT/'outputs';OUT.mkdir(exist_ok=True)
MID='1874553';MATCH=ROOT/'data'/'matches'/MID
SOURCE=OUT/f'P1_canonical_decisions_match_{MID}_ROLES.jsonl'
if not SOURCE.is_file():raise FileNotFoundError(SOURCE)
rows=[json.loads(line) for line in SOURCE.open(encoding='utf-8') if line.strip()]
assert len(rows)==959,f'Expected 959 pilot decisions, got {len(rows)}'
print('Pilot match',MID,'decisions',len(rows))


## 1 — Retrieve only exact required frames

The stream is scanned once; invalid or null source identifiers are counted, not imputed. Required historical keys missing from the same period stay missing. A required `t0` tracking frame is a hard gate because Notebook 05 already established exact linkage.

In [ ]:
def integer(v):
    try:return int(v)
    except (ValueError,TypeError):return None

def xy(p):
    if not isinstance(p,dict):return None
    try:
        x,y=float(p['x']),float(p['y'])
        return (x,y) if math.isfinite(x) and math.isfinite(y) else None
    except (ValueError,KeyError,TypeError):return None

wanted={(int(r['period']),int(r['frame'])-gap) for r in rows for gap in (0,5,10) if int(r['frame'])-gap>=0}
frames={}; null_keys=0
with (MATCH/f'{MID}_tracking_extrapolated.jsonl').open(encoding='utf-8') as stream:
    for lineno,line in enumerate(stream,1):
        f=json.loads(line)
        p,t=integer(f.get('period')),integer(f.get('frame'))
        if p is None or t is None:
            null_keys+=1
            continue
        k=(p,t)
        if k in wanted:
            if k in frames:raise AssertionError(f'Duplicate required tracking key {k}')
            frames[k]=f

present_now=sum((int(r['period']),int(r['frame'])) in frames for r in rows)
assert present_now==len(rows),f'{len(rows)-present_now} possession-start keys missing'
print('Unique requested frames:',len(wanted),'available:',len(frames),'missing:',len(wanted)-len(frames))
print('Invalid-key tracking records:',null_keys)


## 2 — Compute **retrospective displacement rates**, not intrinsic speed

For a player with finite coordinates at `t0` and `t0-h`, calculate `distance / (h/10)` metres per second. For this pilot, the windows are 0.5 and 1.0 s. These are displacement-based average rates, **not** instantaneous speed, maximum sprint speed, or reaction ability.

For each defender at `t0`, retain an individual record with their measured position quality and both historical windows. Record the ball's historical displacement separately; it is **not** a measured pass-travel speed because the earlier ball could be in a different possession context.

In [ ]:
def player_index(frame):
    return {str(p['player_id']):p for p in (frame.get('player_data') or []) if p.get('player_id') is not None}

def rate(current,previous,seconds):
    a,b=xy(current),xy(previous)
    if a is None or b is None:return None
    return math.hypot(a[0]-b[0],a[1]-b[1])/seconds

summary=Counter();samples=[];speed_all=[];speed_observed=[];ball_rates=[]
out=OUT/f'P2_temporal_motion_pilot_match_{MID}.jsonl'
with out.open('w',encoding='utf-8') as dest:
    for r in rows:
        per,t=int(r['period']),int(r['frame'])
        now=frames[(per,t)]
        curr=player_index(now)
        past={h:frames.get((per,t-h)) if t>=h else None for h in (5,10)}
        prev={h:player_index(f) if f is not None else {} for h,f in past.items()}
        defenders=[]
        for quality in ('detected','extrapolated'):
            for defender in r.get('opponents_'+quality,[]):
                pid=str(defender['player_id']);p0=curr.get(pid)
                if p0 is None:raise AssertionError(f'Canonical opponent absent at {(per,t,pid)}')
                d={'player_id':pid,'t0_quality':quality}
                for h in (5,10):
                    ph=prev[h].get(pid)
                    v=rate(p0,ph,h/10)
                    fully_detected=(p0.get('is_detected') is True and ph is not None and ph.get('is_detected') is True)
                    d[f'past_{h}_frames_avg_mps']=v
                    d[f'past_{h}_endpoints_detected']=fully_detected
                    summary[f'defender_{h}_finite']+=v is not None
                    summary[f'defender_{h}_detected_endpoints']+=v is not None and fully_detected
                    if h==5 and v is not None:
                        speed_all.append(v)
                        if fully_detected:speed_observed.append(v)
                        summary['defender_halfsec_rate_gt_15mps']+=v>15
                defenders.append(d)
        b0=now.get('ball_data') or {}
        br={}
        for h in (5,10):
            earlier=(past[h].get('ball_data') or {}) if past[h] is not None else None
            v=rate(b0,earlier,h/10)
            br[str(h)]=v
            summary[f'ball_{h}_finite']+=v is not None
            if h==5 and v is not None:ball_rates.append(v)
        summary['decisions']+=1
        summary['defender_records']+=len(defenders)
        for h in (5,10):
            summary[f'past_{h}_frame_present']+=past[h] is not None
            summary[f'carrier_{h}_past_rate_finite']+=rate(curr.get(str(r['carrier_player_id'])),prev[h].get(str(r['carrier_player_id'])),h/10) is not None
        record={'match_id':MID,'period':per,'frame':t,'possession_event_id':r['possession_event_id'],
          'carrier_player_id':r['carrier_player_id'],'past_defender_motion':defenders,
          'past_ball_displacement_rate_mps':br,'past_frames_available':{str(h):past[h] is not None for h in (5,10)},
          'interpretation':'retrospective_snapshot_displacement_only'}
        dest.write(json.dumps(record,separators=(',',':'))+'\n')

assert summary['decisions']==959
print('Decision counts:',summary['decisions'])
print('Defender records:',summary['defender_records'])
print('Past frame availability:',{h:summary[f'past_{h}_frame_present'] for h in (5,10)})


## 3 — Gate report

Rates above 15 m/s are only a *diagnostic review flag* (not an exclusion or an athlete physiological threshold). Suspicious values may reflect extrapolation, identity/frame issues, tracking error, or genuinely unusual ball/player motion. We are not fitting any interception or passing model here.

In [ ]:
def dist_summary(a):
    if not a:return {'n':0,'median':None,'p95':None,'max':None}
    v=sorted(a)
    return {'n':len(v),'median':statistics.median(v),'p95':v[min(len(v)-1,int(.95*(len(v)-1)))],'max':v[-1]}

gate={'created_utc':datetime.now(timezone.utc).isoformat(),
      'purpose':'Past-only temporal tracking quality pilot; not calibrated player or ball speeds',
      'match_id':MID,'expected_decisions':959,
      'unique_requested_keys':len(wanted),'unique_found_keys':len(frames),
      'unavailable_requested_keys':len(wanted)-len(frames),
      'tracking_records_skipped_due_to_invalid_key':null_keys,
      'counts':dict(summary),
      'defender_halfsec_displacement_all_finite_mps':dist_summary(speed_all),
      'defender_halfsec_displacement_both_endpoints_detected_mps':dist_summary(speed_observed),
      'ball_halfsec_historical_displacement_mps':dist_summary(ball_rates),
      'limitations':[
        'Exact historical same-period frame keys only; no future frames',
        'Player displacement over 0.5s/1s is NOT instantaneous speed or a sprint capability estimate',
        'Detected endpoints do not certify every intermediate observation',
        'Tracking extrapolation can distort displacement and velocity estimates',
        'Historical ball displacement is NOT a prospective pass-travel velocity estimate',
        'Single-match pilot; no inference, pressing classification, or interception outcomes',
        '15m/s player displacement check is a descriptive anomaly flag, not an eligibility cutoff']}
path=OUT/'P2_temporal_motion_gate.json'
path.write_text(json.dumps(gate,indent=2),encoding='utf-8')
print(json.dumps({k:gate[k] for k in ('match_id','unique_requested_keys','unique_found_keys','counts','defender_halfsec_displacement_both_endpoints_detected_mps')},indent=2))
print('UPLOAD:',path)
